# CrisisLens 01: Exploratory data analysis

Goal: understand CrisisMMD v2.0 (agreed-label splits) before modeling. Every finding here should turn into a modeling decision, written in the **Takeaways** cell at the end.

**On Kaggle:** add your CrisisMMD dataset and set `DATA_DIR` below. **Locally:** point it at your `data/` folder.

In [ ]:
# On Kaggle, install the repo first (uncomment):
# !git clone https://github.com/kshitideshpande/crisislens.git
# %pip install -q -e crisislens
import sys
from pathlib import Path

DATA_DIR = Path("/kaggle/input/crisismmd")   # <- change for local runs, e.g. Path("../data")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from crisislens.data import TASKS, SPLITS, load_task, find_image_root

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 140)

## 1. Load all three tasks

In [ ]:
data = {task: load_task(DATA_DIR, task) for task in TASKS}
image_root = find_image_root(DATA_DIR)
print("image root:", image_root)

sizes = pd.DataFrame({task: {s: len(df) for s, df in splits.items()} for task, splits in data.items()}).T
sizes["total"] = sizes.sum(axis=1)
sizes

In [ ]:
data["humanitarian"]["train"].head()

## 2. Class balance

Expect heavy imbalance in the humanitarian task. This decides the loss (class-weighted) and the headline metric (macro-F1).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, task in zip(axes, TASKS):
    tr = data[task]["train"]["label"].value_counts(normalize=True).sort_values()
    tr.rename(lambda s: s.replace("_", " ")).plot.barh(ax=ax, color="#4C72B0")
    ax.set_title(f"{task}: train label share")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    for i, v in enumerate(tr.values):
        ax.text(v, i, f" {v:.1%}", va="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "class_balance.png", dpi=150); plt.show()

for task in TASKS:
    vc = data[task]["train"]["label"].value_counts()
    print(f"{task:<13} imbalance ratio (largest/smallest) = {vc.max() / vc.min():.1f}x")

## 3. Do splits match? (train vs. dev vs. test)

If the label mix differs a lot between splits, dev scores won't predict test scores.

In [ ]:
rows = []
for task, splits in data.items():
    for s, df in splits.items():
        for label, share in df["label"].value_counts(normalize=True).items():
            rows.append({"task": task, "split": s, "label": label, "share": share})
mix = pd.DataFrame(rows).pivot_table(index=["task", "label"], columns="split", values="share")[list(SPLITS)]
mix.style.format("{:.1%}").background_gradient(axis=None, cmap="Blues")

## 4. Events

Seven disasters. Event mix matters for the later leave-one-disaster-out test, and labels may differ a lot by disaster type.

In [ ]:
hum = pd.concat(data["humanitarian"].values(), ignore_index=True)
ct = pd.crosstab(hum["event_name"], hum["label"], normalize="index")
ct.columns = [c.replace("_", " ") for c in ct.columns]
plt.figure(figsize=(11, 4.5))
sns.heatmap(ct, annot=True, fmt=".0%", cmap="Blues", cbar=False)
plt.title("Humanitarian label mix by event (all splits)"); plt.ylabel(""); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig(FIG_DIR / "labels_by_event.png", dpi=150); plt.show()

hum["event_name"].value_counts().to_frame("rows")

## 5. Text

Token lengths decide `max_length` for BERTweet (Week 2). Also check how many rows share the same tweet (one tweet, several images).

In [ ]:
tr = data["humanitarian"]["train"].copy()
tr["n_words"] = tr["text"].str.split().str.len()
print(tr["n_words"].describe(percentiles=[.5, .9, .95, .99]).round(1))

plt.figure(figsize=(8, 3.5))
sns.histplot(tr["n_words"], bins=40)
plt.title("Words per tweet (train)"); plt.tight_layout(); plt.savefig(FIG_DIR / "text_length.png", dpi=150); plt.show()

dup = tr.groupby("tweet_id").size()
print(f"rows={len(tr)}  unique tweets={len(dup)}  tweets with >1 image={(dup > 1).sum()}")

In [ ]:
# Same tweet text in train and test? (leakage check: retweets / copy-paste posts)
for task in TASKS:
    train_txt = set(data[task]["train"]["text"].str.lower())
    test = data[task]["test"]
    overlap = test["text"].str.lower().isin(train_txt).mean()
    print(f"{task:<13} test rows whose exact text appears in train: {overlap:.1%}")

In [ ]:
# Most frequent hashtags per humanitarian class
tags = (tr.assign(tag=tr["text"].str.lower().str.findall(r"#\w+")).explode("tag").dropna(subset=["tag"]))
tags.groupby("label")["tag"].agg(lambda s: ", ".join(s.value_counts().head(6).index))

## 6. Example posts per class

In [ ]:
for label, g in tr.groupby("label"):
    print(f"\n--- {label} ---")
    for t in g["text"].sample(3, random_state=0):
        print("  •", t[:160])

## 7. Images

Skips gracefully if images aren't downloaded yet. Note: some disaster images can be distressing.

In [ ]:
def show_grid(df, label, n=6):
    paths = df[df["label"] == label]["image"].sample(min(n, (df["label"] == label).sum()), random_state=1)
    fig, axes = plt.subplots(1, len(paths), figsize=(3 * len(paths), 3))
    for ax, p in zip(np.atleast_1d(axes), paths):
        try:
            ax.imshow(Image.open(image_root / p).convert("RGB"))
        except Exception as e:
            ax.text(0.5, 0.5, "missing", ha="center")
        ax.axis("off")
    fig.suptitle(label.replace("_", " ")); plt.tight_layout(); plt.show()

if image_root is None:
    print("Images not found; download CrisisMMD_v2.0.tar.gz to see this section.")
else:
    for label in sorted(data["damage"]["train"]["label"].unique()):
        show_grid(data["damage"]["train"], label)

In [ ]:
if image_root is not None:
    sample = data["humanitarian"]["train"]["image"].sample(300, random_state=0)
    sizes = []
    for p in sample:
        try:
            with Image.open(image_root / p) as im:
                sizes.append(im.size)
        except Exception:
            pass
    s = pd.DataFrame(sizes, columns=["width", "height"])
    print(s.describe().round(0))
    print(f"unreadable images in sample: {len(sample) - len(s)}")

## Takeaways (fill in after running)

Write one line each, with the actual numbers:

1. **Imbalance:** humanitarian imbalance ratio is __x → use class-weighted loss, report macro-F1.
2. **Splits:** label mix across train/dev/test is (similar / different) → dev is (a good / a weak) proxy for test.
3. **Events:** largest event is __ (__% of rows) → leave-one-disaster-out test will matter for __.
4. **Text:** 95th percentile is __ words → BERTweet `max_length = __` tokens.
5. **Duplicates/leakage:** __% of test texts appear in train → (fine / note it as a limitation).
6. **Images:** typical size __, __ unreadable → resize to 224 for CLIP; drop/replace unreadable.